# Task 1: Data Exploration and Enrichment

Objective: understand the unified schema, explore starter records, and document enrichment additions for Access and Usage forecasting.

**Schema principle:** events are *not* pre-assigned to pillars. Effects are modeled only through `impact_link` rows via `parent_id`.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data_loader import load_unified_data, load_impact_sheet, load_reference_codes

data = load_unified_data()
impacts = load_impact_sheet()
refs = load_reference_codes()
print('Unified shape:', data.shape)
print('Impact shape:', impacts.shape)
print('Reference codes shape:', refs.shape)
data['record_type'].value_counts()


## Schema checks
Count by `record_type`, `pillar`, `source_type`, and `confidence`. List indicators and events.


In [ ]:
obs = data[data.record_type=='observation'].copy()
events = data[data.record_type=='event'].copy()
targets = data[data.record_type=='target'].copy()

print('By record_type:\n', data.record_type.value_counts(), '\n')
print('By pillar (non-null):\n', data.pillar.value_counts(dropna=True), '\n')
print('By source_type:\n', data.source_type.value_counts(dropna=False), '\n')
print('By confidence:\n', data.confidence.value_counts(dropna=False), '\n')
print('Temporal range (observations):', obs.observation_date.min(), '->', obs.observation_date.max())
print('\nIndicators:\n', obs.indicator_code.value_counts())
print('\nEvents:\n', events[['record_id','category','indicator','observation_date']].sort_values('observation_date'))
print('\nImpact links (parent -> indicator):')
print(impacts[['record_id','parent_id','pillar','related_indicator','impact_direction','impact_magnitude','impact_estimate','lag_months']])


## Enrichment documentation
See `data_enrichment_log.md` for source_url, original_text, confidence, collected_by, collection_date, and notes for every addition.


In [ ]:
enriched_ids = [f'REC_{i:04d}' for i in range(36,46)] + ['EVT_0011','EVT_0012'] + [f'IMP_{i:04d}' for i in range(16,20)]
print('New observation/event rows present:', data.record_id.isin(enriched_ids).sum())
print('New impact rows present:', impacts.record_id.isin(enriched_ids).sum())
data[data.record_id.isin(enriched_ids)][['record_id','record_type','pillar','category','indicator_code','value_numeric','observation_date','confidence','collected_by']]
